# MADLAD-400 3B · Explorer V100 · patched single-GPU run (JAX disabled)

**Run All once.** Designed for your allocated **Tesla V100 32 GB**, four CPUs, and an Explorer Jupyter session. The notebook uses a private virtual environment with PyTorch from Explorer and disables incompatible inherited JAX/Flax imports (no changes to shared Anaconda), loads original XSum, performs **English → Spanish → English**, checkpoints after **every document**, calculates slide metrics, and saves CSVs. No gated Hugging Face token required.

**Default: 200 documents** for a deadline-friendly pilot. After success, change `MAX_ITEMS = 200` to `None` to run all available original XSum records. A run may outlast your session; results persist for resumption as long as the output directory persists. **The full run is not guaranteed to finish in one session.**

Metrics are for the MADLAD outputs only; do not compare them as a matched statistical test against Qwen unless the same IDs and scoring implementations are used. Source: Ship of Theseus GitHub repository.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time

MODEL_ID = 'google/madlad400-3b-mt'
MAX_ITEMS = 200                # Pilot: 200. Full XSum: None. Smoke test: 12.
CHUNK_BUDGET = 300             # Input token limit, includes language prefix.
BATCH_SIZE = 4                 # Chunks per generate call; one V100 GPU.
SAVE_EVERY = 1                # Persist each completed document.
QWEN_CHAIN_CSV = None         # Optional absolute path if you later obtain it.

# Store data outside /scratch unless you explicitly prefer temporary scratch storage.
PROJECT = Path.home() / 'degree_zero_madlad_v100'
ENV = PROJECT / '.venv'
OUT = PROJECT / 'output'
PROJECT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print('Python kernel:', sys.executable)
print('Project:', PROJECT)
print('Target sample:', MAX_ITEMS or 'all original XSum')
# nvidia-smi checks the GPU even if this Jupyter kernel has broken sklearn/numpy imports.
check = subprocess.run(['nvidia-smi','-L'], text=True, capture_output=True)
print('GPU detection:', check.stdout.strip() or check.stderr.strip())
if check.returncode != 0 or 'GPU 0:' not in check.stdout:
    raise RuntimeError('No NVIDIA GPU detected: launch Explorer Jupyter on a GPU allocation first.')

## 1. Prepare an isolated environment (once)

This uses the system's **existing CUDA-enabled PyTorch** without reinstalling its enormous GPU wheels, but installs matching NumPy / SciPy / scikit-learn in the project's own virtual environment. The worker sets `PYTHONNOUSERSITE=1` so your previously installed `~/.local` packages cannot leak in. Initial installation/download may take several minutes. Do **not** run `%pip install -U` on the shared Jupyter kernel.

In [ ]:
import venv
if not (ENV / 'bin' / 'python').exists():
    venv.EnvBuilder(system_site_packages=True, with_pip=True).create(str(ENV))
PY = str(ENV / 'bin' / 'python')
worker_env = dict(os.environ)
worker_env['PYTHONNOUSERSITE'] = '1'
# The base Anaconda environment exposes incompatible JAX packages.
# Transformers must not auto-import JAX/Flax or TensorFlow for this PyTorch model.
worker_env['USE_FLAX'] = '0'
worker_env['USE_TF'] = '0'
worker_env['USE_TORCH'] = '1'
worker_env['PYTHONUNBUFFERED'] = '1'
worker_env['TOKENIZERS_PARALLELISM'] = 'false'
scratch_cache = Path('/scratch') / os.environ.get('USER','') / 'madlad_hf_cache'
cache_home = scratch_cache if scratch_cache.parent.exists() and os.access(scratch_cache.parent,os.W_OK) else PROJECT / 'hf_cache'
worker_env['HF_HOME'] = str(cache_home)
worker_env['HF_HUB_DISABLE_TELEMETRY'] = '1'
Path(worker_env['HF_HOME']).mkdir(exist_ok=True, parents=True)

# Pin a coherent NumPy/scikit-learn stack; keep torch from Explorer's base environment.
packages = [
    'numpy==1.26.4', 'scipy==1.13.1', 'scikit-learn==1.5.2',
    'transformers==4.46.3', 'accelerate==1.2.1',
    'sentencepiece==0.2.0', 'pandas==2.2.3',
    'sacrebleu==2.5.1', 'rouge-score==0.1.2', 'tqdm==4.67.1',
]
marker = ENV / '.requirements_ok_v2'
if not marker.exists():
    print('Installing isolated dependencies. This may take a few minutes...')
    subprocess.run([PY,'-m','pip','install','--no-cache-dir','--upgrade']+packages,
                   env=worker_env,check=True)
    marker.touch()

probe = """import numpy, scipy, sklearn, torch, transformers, pandas
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.models.t5.modeling_t5 import T5ForConditionalGeneration
print('numpy', numpy.__version__, 'sklearn',sklearn.__version__, 'torch',torch.__version__, 'transformers',transformers.__version__)
print('CUDA',torch.cuda.is_available(), 'GPUs',torch.cuda.device_count())
"""
run = subprocess.run([PY,'-c',probe],env=worker_env,text=True,capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr[-4000:])
    raise RuntimeError('Private environment validation failed; no GPU experiment started.')
if 'CUDA True' not in run.stdout:
    raise RuntimeError('Private environment cannot see CUDA. Check Explorer GPU allocation.')

## 2. Write the standalone inference script

The worker owns loading, chunking, translation, validation, checkpointing and metric generation. Progress logs are printed live in the next cell. Outputs persist after a kernel restart. It refuses to silently switch models or input texts inside an existing checkpoint directory.

In [ ]:
WORKER = PROJECT / 'madlad_run.py'
WORKER.write_text('import argparse, csv, hashlib, json, os, re, time, traceback\nfrom collections import Counter\nfrom pathlib import Path\nfrom urllib.request import urlopen\nimport pandas as pd\nimport numpy as np\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSeq2SeqLM\nfrom sacrebleu.metrics import BLEU, CHRF\nfrom rouge_score import rouge_scorer\n\np=argparse.ArgumentParser()\np.add_argument(\'--out\',required=True)\np.add_argument(\'--max-items\',default=\'200\')\np.add_argument(\'--model\',required=True)\np.add_argument(\'--budget\',type=int,default=300)\np.add_argument(\'--batch\',type=int,default=4)\np.add_argument(\'--qwen-csv\',default=\'\')\na=p.parse_args()\nout=Path(a.out); out.mkdir(parents=True,exist_ok=True)\nsource_url=\'https://raw.githubusercontent.com/tripto03/Ship_of_theseus_paraphrased_copus/main/paraphrased_datasets/xsum_paraphrased.csv\'\n\nprint(\'Preparing original XSum sources...\', flush=True)\nif a.qwen_csv:\n    q=pd.read_csv(a.qwen_csv)\n    if not {\'item_id\',\'S\'}.issubset(q.columns): raise ValueError(\'Qwen chain requires item_id and S columns\')\n    items=q[[\'item_id\',\'S\']].copy()\n    parts=items.item_id.astype(str).str.split(\'|\',n=2,expand=True)\n    items[[\'domain\',\'source\',\'key\']]=parts\n    if not items.domain.eq(\'xsum\').all(): raise ValueError(\'Not all Qwen records are XSum\')\nelse:\n    raw=pd.read_csv(source_url)\n    required={\'version_name\',\'key\',\'source\',\'text\'}\n    if not required.issubset(raw): raise ValueError(f\'Missing input fields: {required-set(raw.columns)}\')\n    items=(raw.loc[raw.version_name.eq(\'original\'),[\'key\',\'source\',\'text\']]\n             .dropna(subset=[\'text\']).drop_duplicates([\'key\',\'source\'])\n             .rename(columns={\'text\':\'S\'}).copy())\n    items[\'domain\']=\'xsum\'\n    items[\'item_id\']=items.domain+\'|\'+items.source.astype(str)+\'|\'+items.key.astype(str)\n    if len(items)!=3227:raise RuntimeError(f\'Expected 3227 source items, found {len(items)}; repository may have changed\')\nitems=items.sort_values([\'source\',\'item_id\']).reset_index(drop=True)\n# Balanced deterministic round-robin by source, so the 200-doc pilot covers all 7 origins.\nitems[\'_source_pos\']=items.groupby(\'source\').cumcount()\nitems=items.sort_values([\'_source_pos\',\'source\',\'item_id\']).drop(columns=[\'_source_pos\']).reset_index(drop=True)\nif items.item_id.duplicated().any() or items.S.astype(str).str.strip().eq(\'\').any():raise ValueError(\'Invalid source IDs/text\')\nif a.max_items.lower()!=\'none\':items=items.iloc[:int(a.max_items)].copy()\nitems[\'words\']=items.S.astype(str).str.split().str.len()\nmanifest=out/\'input_manifest.csv\'\n# Keep an immutable manifest and allow prefix expansion (200 -> 3227) without mixing sources.\nif manifest.exists():\n    past=pd.read_csv(manifest).fillna(\'\')\n    now=items.fillna(\'\')\n    n=min(len(past),len(now))\n    if (past.item_id.astype(str).iloc[:n].tolist()!=now.item_id.astype(str).iloc[:n].tolist() or\n        past.S.astype(str).iloc[:n].tolist()!=now.S.astype(str).iloc[:n].tolist()):\n        raise RuntimeError(\'Input dataset changed; use a new output directory\')\n    if len(items)>len(past):items.to_csv(manifest,index=False)\nelse:items.to_csv(manifest,index=False)\nprint(f\'Selected {len(items)} documents from {items.source.nunique()} sources; median words={items.words.median():.0f}\',flush=True)\n\ncheckpoint=out/\'chain_es_madlad4003b.csv\'\nif checkpoint.exists():\n    done=pd.read_csv(checkpoint,keep_default_na=False)\n    if done.item_id.duplicated().any():raise RuntimeError(\'Duplicate IDs in checkpoint\')\n    known=set(done.item_id.astype(str))\n    if known-set(items.item_id.astype(str)):\n        print(\'Previously saved results include documents beyond the current sample; keeping them intact\',flush=True)\n    already=len(known & set(items.item_id.astype(str)))\nelse:\n    known=set();already=0\nprint(f\'Already recorded {already}/{len(items)}; existing failures will NOT rerun automatically.\',flush=True)\n\nif already<len(items):\n    print(\'Loading MADLAD model...\',flush=True)\n    tokenizer=AutoTokenizer.from_pretrained(a.model)\n    model=AutoModelForSeq2SeqLM.from_pretrained(a.model,torch_dtype=torch.float16,low_cpu_mem_usage=True).to(\'cuda:0\')\n    model.eval()\n    print(\'Loaded on\',torch.cuda.get_device_name(0),flush=True)\n    def token_count(s,lang):\n        return len(tokenizer(f\'<2{lang}> \'+s,add_special_tokens=True).input_ids)\n    def split_chunks(text,lang):\n        pieces=[]\n        for sentence in re.split(r\'(?<=[.!?])\\s+\',str(text).strip()):\n            sentence=sentence.strip()\n            if not sentence:continue\n            if token_count(sentence,lang)<=a.budget:pieces.append(sentence);continue\n            part=\'\'\n            for word in sentence.split():\n                trial=(part+\' \'+word).strip()\n                if token_count(trial,lang)<=a.budget:part=trial\n                else:\n                    if part:pieces.append(part)\n                    if token_count(word,lang)>a.budget:raise RuntimeError(\'one_word_exceeds_budget\')\n                    part=word\n            if part:pieces.append(part)\n        result=[];cur=\'\'\n        for part in pieces:\n            candidate=(cur+\' \'+part).strip()\n            if token_count(candidate,lang)<=a.budget:cur=candidate\n            else:\n                if cur:result.append(cur)\n                cur=part\n        if cur:result.append(cur)\n        if not result:raise RuntimeError(\'no_input_chunks\')\n        return result\n    def translate(text,lang):\n        segments=split_chunks(text,lang)\n        outputs=[];truncated=False\n        eos=model.generation_config.eos_token_id\n        if eos is None:eos=tokenizer.eos_token_id\n        eos_set=set(eos) if isinstance(eos,(tuple,list,set)) else {int(eos)}\n        for i in range(0,len(segments),a.batch):\n            batch=segments[i:i+a.batch]\n            enc=tokenizer([f\'<2{lang}> \'+x for x in batch],padding=True,return_tensors=\'pt\',truncation=False).to(\'cuda:0\')\n            with torch.inference_mode():\n                gen=model.generate(**enc,num_beams=1,do_sample=False,max_new_tokens=512)\n            for ids in gen:\n                decoded=tokenizer.decode(ids,skip_special_tokens=True).strip()\n                if not decoded:raise RuntimeError(\'empty_translation\')\n                if not any(int(t) in eos_set for t in ids.tolist()):truncated=True\n                outputs.append(decoded)\n        return \' \'.join(outputs),len(segments),truncated\n\n    write_header=not checkpoint.exists() or checkpoint.stat().st_size==0\n    fields=list(items.columns)+[\'F\',\'B_F\',\'bad\',\'bad_reason\',\'chunks_en\',\'chunks_es\',\'seconds\']\n    with checkpoint.open(\'a\',newline=\'\',encoding=\'utf-8\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=fields,extrasaction=\'ignore\')\n        if write_header:writer.writeheader();stream.flush();os.fsync(stream.fileno())\n        tstart=time.monotonic();added=0\n        for record in items.to_dict(\'records\'):\n            key=str(record[\'item_id\'])\n            if key in known:continue\n            started=time.monotonic()\n            data=dict(record);data.update(F=\'\',B_F=\'\',bad=True,bad_reason=\'\',chunks_en=0,chunks_es=0,seconds=0)\n            try:\n                es,n_en,cap_en=translate(record[\'S\'],\'es\')\n                en,n_es,cap_es=translate(es,\'en\')\n                reasons=[]\n                if cap_en or cap_es:reasons.append(\'generation_cap\')\n                ratio=len(en.split())/max(1,len(str(record[\'S\']).split()))\n                if ratio<.35:reasons.append(\'severe_compression\')\n                if ratio>3:reasons.append(\'severe_expansion\')\n                data.update(F=es,B_F=en,bad=bool(reasons),bad_reason=\';\'.join(reasons),chunks_en=n_en,chunks_es=n_es)\n            except Exception as e:\n                data[\'bad_reason\']=f\'{type(e).__name__}: {str(e)[:180]}\'\n            data[\'seconds\']=round(time.monotonic()-started,2)\n            writer.writerow(data);stream.flush();os.fsync(stream.fileno())\n            known.add(key);added+=1\n            if added%5==0 or data[\'bad\']:\n                print(f\'{added} new / {len(items)} total | last {data["seconds"]:.1f}s | bad={data["bad"]} | elapsed={(time.monotonic()-tstart)/60:.1f} min\',flush=True)\n    del model\n    torch.cuda.empty_cache()\n\nprint(\'Preparing evaluation metrics...\',flush=True)\nresult=pd.read_csv(checkpoint,keep_default_na=False)\nsubset=set(items.item_id.astype(str))\nresult=result[result.item_id.astype(str).isin(subset)].copy()\nresult[\'bad\']=result.bad.astype(str).str.lower().eq(\'true\')\nresult[\'valid\']=(~result.bad & result.F.astype(str).str.strip().ne(\'\') & result.B_F.astype(str).str.strip().ne(\'\'))\nvalid=result[result.valid].copy()\nbleu=BLEU(effective_order=True)\nchrf=CHRF()\nrouge=rouge_scorer.RougeScorer([\'rougeL\'],use_stemmer=True)\nfunc=set(\'a an the and or but if of to in on for with at from by is are was were be been being it its this that these those as not do does did have has had he she they them his her their we us our you your i me my\'.split())\ndef words(s):return re.findall(r"\\b\\w+\\b",str(s).lower())\ndef word_retention(source,hyp):\n    a=Counter(words(source));b=Counter(words(hyp))\n    return sum(min(count,b.get(w,0)) for w,count in a.items())/max(sum(a.values()),1)\ndef function_cos(source,hyp):\n    x=Counter(w for w in words(source) if w in func);y=Counter(w for w in words(hyp) if w in func)\n    dot=sum(x[w]*y[w] for w in func)\n    nx=sum(x[w]**2 for w in func)**.5;ny=sum(y[w]**2 for w in func)**.5\n    return dot/(nx*ny) if nx and ny else float(\'nan\')\ndef punctuation_f1(source,hyp):\n    x=Counter(ch for ch in str(source) if ch in \'.,!?;:\\\\"\\\'()-\')\n    y=Counter(ch for ch in str(hyp) if ch in \'.,!?;:\\\\"\\\'()-\')\n    hit=sum(min(v,y.get(k,0)) for k,v in x.items())\n    return 2*hit/max(sum(x.values())+sum(y.values()),1)\nrows=[]\nfor r in valid.itertuples(index=False):\n    s=str(r.S);h=str(r.B_F)\n    rows.append(dict(item_id=r.item_id,source=r.source,words=r.words,seconds=r.seconds,\n       bleu=bleu.sentence_score(h,[s]).score/100,\n       rougeL=rouge.score(s,h)[\'rougeL\'].fmeasure,\n       chrf=chrf.sentence_score(h,[s]).score/100,\n       word_retention=word_retention(s,h),\n       function_word_cosine=function_cos(s,h),\n       punctuation_f1=punctuation_f1(s,h),\n       exact_match=float(s.strip()==h.strip())))\nmetrics=pd.DataFrame(rows)\nmetrics.to_csv(out/\'matched_item_metrics.csv\',index=False)\nif not metrics.empty:\n    summary=metrics.drop(columns=[\'item_id\',\'source\']).select_dtypes(include=\'number\').agg([\'mean\',\'median\',\'std\']).T\n    summary.to_csv(out/\'slide_metrics_summary.csv\')\n    print(\'METRIC MEDIANS (higher = greater preservation):\',flush=True)\n    print(summary[\'median\'].round(3).to_string(),flush=True)\nprint(f\'RUN COMPLETE: selected={len(items)} recorded={len(result)} valid={len(valid)} bad={len(result)-len(valid)}\',flush=True)\nprint(\'Files:\', out/\'chain_es_madlad4003b.csv\', out/\'matched_item_metrics.csv\',out/\'slide_metrics_summary.csv\',sep=\'\\n\',flush=True)\n', encoding='utf-8')
print('Worker ready:', WORKER)

## 3. Run the experiment (this cell stays busy while the GPU works)

**This is the only long-running cell.** It prints progress every five documents and writes each document to disk immediately. If the Jupyter session expires, relaunch with a GPU and rerun the notebook; completed IDs are skipped. If you interrupt this cell, stop it before starting another run to prevent duplicate GPU workers.

In [ ]:
cmd = [PY, '-u', str(WORKER), '--out', str(OUT), '--model', MODEL_ID,
       '--max-items', 'None' if MAX_ITEMS is None else str(MAX_ITEMS),
       '--budget', str(CHUNK_BUDGET), '--batch', str(BATCH_SIZE)]
if QWEN_CHAIN_CSV:cmd += ['--qwen-csv',str(QWEN_CHAIN_CSV)]
print('Starting 1-GPU experiment. Checkpoints:', OUT / 'chain_es_madlad4003b.csv', flush=True)
print('Command:', ' '.join(cmd[:3]), '...')
# Blocking foreground execution makes Run All safe: metrics happen only after translations complete.
with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                      text=True,bufsize=1,env=worker_env,cwd=PROJECT) as proc:
    try:
        for line in proc.stdout:
            print(line,end='',flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        raise
    rc=proc.wait()
if rc:
    raise RuntimeError(f'Worker stopped with exit code {rc}. Read traceback above; checkpoint remains intact.')

## 4. Show results without importing the damaged global scientific stack

This reads the generated metrics using the isolated Python environment, then prints a convenient summary. Notebook outputs are saved under `~/degree_zero_madlad_v100/output`.

**Interpretation:** Scores represent preservation relative to original T0; they are not the same normalization as Shushruth's `metrics_summary.json`. BLEU and ROUGE-L here are sentence/document-level averages on this pilot, not corpus BLEU.

In [ ]:
show = r"""from pathlib import Path
import pandas as pd
p=Path(__import__('sys').argv[1]);chain=p/'chain_es_madlad4003b.csv';metrics=p/'slide_metrics_summary.csv'
if chain.exists():
 df=pd.read_csv(chain)
 bad=df.bad.astype(str).str.lower().eq('true')
 print('Records:',len(df),'Valid:',int((~bad).sum()),'Flagged:',int(bad.sum()))
if metrics.exists():
 df=pd.read_csv(metrics,index_col=0)
 print('\nMetric summary:')
 print(df.round(3).to_string())
else: print('Metrics not available yet')
"""
r = subprocess.run([PY,'-c',show,str(OUT)],env=worker_env,text=True,capture_output=True)
print(r.stdout or r.stderr)
print('Your files live in:',OUT)

### Saved outputs
- `chain_es_madlad4003b.csv`: full original, Spanish, back-translation, completion flags, elapsed seconds
- `matched_item_metrics.csv`: valid-item BLEU, ROUGE-L, chrF, word retention, function-word cosine, punctuation F1
- `slide_metrics_summary.csv`: mean/median/std for presentation
- `input_manifest.csv`: selected XSum IDs and source texts

**Cautions:** This model may hallucinate or mistranslate facts; the simple length and generation-cap flags do not prove translation fidelity. Semantic E5/BERTScore are deliberately not included in this quick, robust inference run; those need a separate evaluation step. Results are a deterministic source-balanced XSum pilot, not a random representative sample. Check your Explorer job time remaining before starting a long run.